In [2]:
!python -V

Python 3.9.23


In [24]:
import pandas as pd
import numpy as np
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn

In [27]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_squared_error


In [28]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/workspaces/mlops-zoomcamp/03-training/experiment_tracking/mlruns/1', creation_time=1753506252722, experiment_id='1', last_update_time=1753506252722, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [29]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [30]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')

In [31]:
len(df_train), len(df_val)

(73908, 61921)

In [32]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [33]:
def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
        df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [34]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')

In [35]:
len(df_train), len(df_val)

(73908, 61921)

In [36]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [38]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [39]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [45]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)


rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("RMSE:", rmse)

RMSE: 7.758715209663881


In [64]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [49]:
with mlflow.start_run():

    mlflow.set_tag("developer", "Phan Dang Ha")

    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.parquet")

    alpha = 0.1
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    # rmse = root_mean_squared_error(y_val, y_pred)
    # rmse = mean_squared_error(y_val, y_pred, squared=False)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

In [50]:
import xgboost as xgb

In [51]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [52]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [54]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=30
        )
        y_pred = booster.predict(valid)
        # rmse = root_mean_squared_error(y_val, y_pred)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [55]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=30,
    trials=Trials()
)

  0%|          | 0/30 [00:00<?, ?trial/s, best loss=?]

/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:46:59] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:10.51898                          
[1]	validation-rmse:9.29548                           
[2]	validation-rmse:8.43136                           
[3]	validation-rmse:7.82921                           
[4]	validation-rmse:7.41736                           
[5]	validation-rmse:7.13691                           
[6]	validation-rmse:6.94698                           
[7]	validation-rmse:6.81533                           
[8]	validation-rmse:6.72334                           
[9]	validation-rmse:6.65862                           
[10]	validation-rmse:6.61034                          
[11]	validation-rmse:6.57721                          
[12]	validation-rmse:6.55272                          
[13]	validation-rmse:6.53373                          
[14]	validation-rmse:6.51867                          
[15]	validation-rmse:6.50805                          
[16]	validation-rmse:6.49930                          
[17]	validation-rmse:6.49253                          
[18]	valid

/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:49:29] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:7.78708                                                       
[1]	validation-rmse:6.91190                                                       
[2]	validation-rmse:6.72478                                                       
[3]	validation-rmse:6.66782                                                       
[4]	validation-rmse:6.63860                                                       
[5]	validation-rmse:6.62354                                                       
[6]	validation-rmse:6.61798                                                       
[7]	validation-rmse:6.61275                                                       
[8]	validation-rmse:6.60535                                                       
[9]	validation-rmse:6.60032                                                       
[10]	validation-rmse:6.59593                                                      
[11]	validation-rmse:6.59303                                                      
[12]

...

[0]	validation-rmse:10.18499                                                    
[1]	validation-rmse:8.85405                                                     
[2]	validation-rmse:8.00976                                                     
[3]	validation-rmse:7.48842                                                     
[4]	validation-rmse:7.16653                                                     
[5]	validation-rmse:6.96690                                                     
[6]	validation-rmse:6.84285                                                     
[7]	validation-rmse:6.76405                                                     
[8]	validation-rmse:6.71097                                                     
[9]	validation-rmse:6.67745                                                     
[10]	validation-rmse:6.64807                                                    
[11]	validation-rmse:6.63156                                                    
[12]	validation-rmse:6.61884

/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:37:46] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:10.25983                                                    
[1]	validation-rmse:8.95250                                                     
[2]	validation-rmse:8.09877                                                     
[3]	validation-rmse:7.55784                                                     
[4]	validation-rmse:7.22023                                                     
[5]	validation-rmse:7.00593                                                     
[6]	validation-rmse:6.86900                                                     
[7]	validation-rmse:6.78238                                                     
[8]	validation-rmse:6.72191                                                     
[9]	validation-rmse:6.68137                                                     
[10]	validation-rmse:6.65273                                                    
[11]	validation-rmse:6.63208                                                    
[12]	validation-rmse:6.61692

/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:39:13] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[1]	validation-rmse:9.79167                                                     
[2]	validation-rmse:8.99675                                                     
[3]	validation-rmse:8.40208                                                     
[4]	validation-rmse:7.96176                                                     
[5]	validation-rmse:7.63681                                                     
[6]	validation-rmse:7.40025                                                     
[7]	validation-rmse:7.22915                                                     
[8]	validation-rmse:7.10432                                                     
[9]	validation-rmse:7.01116                                                     
[10]	validation-rmse:6.94242                                                    
[11]	validation-rmse:6.89116                                                    
[12]	validation-rmse:6.85400                                                    
[13]	validation-rmse:6.82644

In [57]:
import os
os.path.exists("models/preprocessor.b")


False

In [61]:
from sklearn.feature_extraction import DictVectorizer
import pickle

dv = DictVectorizer()
dv.fit_transform(data_dict)

# Đảm bảo thư mục tồn tại
os.makedirs("models", exist_ok=True)

# Lưu preprocessor
with open("models/preprocessor.b", "wb") as f_out:
    pickle.dump(dv, f_out)


NameError: name 'data_dict' is not defined

In [60]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.parquet")
        mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.parquet")
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mlflow.log_metric("rmse", rmse)

FileNotFoundError: [Errno 2] No such file or directory: 'models/preprocessor.b'